# Leftover Lens: one real photo through the whole pipeline (Colab, T4)

First real run of the measure and match stages. Segmentation is our trained ONNX model;
depth is Depth Anything 3 (DA3-Small); the menu match is SigLIP 2 against the Village
menu for 14 Nov 2026 (the evening the photos were taken, from `tests/fixtures`).

Needs in My Drive: `leftover-lens.zip`, `leftover-photos.zip`, `leftover-runs/village-v2.onnx`
(the 1024 px model; change `ONNX` below to `village-v1.onnx` to compare).

**What this can and cannot show.** We have no weighed plates, so nothing here says the grams
are *right*. It checks that every model loads and runs on real photos, and that the numbers
are physically sane: the camera distance is plausible, the bare plate comes out flat, an
empty plate measures about 0 ml, and changing the focal-length source moves the answer by a
visible, explainable amount.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
ONNX = "/content/drive/MyDrive/leftover-runs/village-v2.onnx"
REPO_ZIP = "/content/drive/MyDrive/leftover-lens.zip"
PHOTOS_ZIP = "/content/drive/MyDrive/leftover-photos.zip"
DA3_REPO = "https://github.com/ByteDance-Seed/Depth-Anything-3"
!rm -rf /content/leftover-lens && unzip -q -o {REPO_ZIP} -d /content
!rm -rf /content/photos && mkdir -p /content/photos && unzip -q -o {PHOTOS_ZIP} -d /content/photos
%cd /content/leftover-lens
!pip -q install -e ".[vision]" transformers sentencepiece protobuf
!rm -rf /content/da3 && git clone -q --depth 1 {DA3_REPO} /content/da3
!pip -q install -e /content/da3
!python -c "import depth_anything_3, onnxruntime, transformers; print('ok')"

## Load the three models and the 14 Nov menu

In [ ]:
import shutil
from datetime import date
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from api.pipeline import merge_by_category
from depth.estimator import DepthEstimator
from depth.volume import measure_food_volumes
from menu import load_daily_menu
from menu.importer import cache_path
from menu.matcher import MenuMatcher, SiglipEmbedder
from portion.grams import estimate_grams
from vision.infer import Segmenter

DAY, MEAL, DATA_DIR = date(2026, 11, 14), "Dinner", Path("/content/data")
cached = cache_path(DATA_DIR / "menus", "university-village", DAY)
cached.parent.mkdir(parents=True, exist_ok=True)
shutil.copy("tests/fixtures/village_2026-11-14.json", cached)
menu = load_daily_menu(DAY, data_dir=DATA_DIR)

segmenter = Segmenter(Path(ONNX))
estimator = DepthEstimator()  # DA3-Small, downloads on first use
matcher = MenuMatcher(menu, SiglipEmbedder(), meal=MEAL)
print("models ready;", len(menu.meals), "meals on the 14 Nov menu")

## Run the photos

Held-out photos (never trained on) plus a rice plate. Crops are skipped: cropping changes the
field of view, so the focal length from the phone no longer applies.

Two focal-length sources are compared, because volume scales one-for-one with focal length
(`docs/MEASUREMENT.md`):
* **DA3**: the focal length Depth Anything 3 predicts from the image;
* **phone**: the iPhone main camera's 26 mm (35 mm-equivalent) lens, i.e. `26 / 36 x` the
  image's long side in pixels. Approximate: the phone's actual capture and our 1600 px resize
  are assumed to keep the full field of view.

In [ ]:
NAMES = [
    "empty_p1_clean_01.jpg",
    "fries_p4_half_04.jpg",
    "fries_p4_full_01.jpg",
    "asparagus+chicken_p1_full_01.jpg",
    "rice-pilaf_p1_half_01.jpg",
    "chicken+rice-pilaf_p1_half_03.jpg",
]


def phone_focal(image):
    return 26.0 / 36.0 * max(image.shape[:2])


rows = []
for name in NAMES:
    path = next(Path("/content/photos").rglob(name), None)
    if path is None:
        print("missing", name)
        continue
    image = cv2.imread(str(path))
    found, plate = segmenter.segment(image)
    regions = merge_by_category(found)
    if plate is None:
        print(name, "-> no plate found")
        continue
    pred = estimator.predict_full(image)
    print(
        f"\n{name}: {len(regions)} food classes, DA3 focal {pred.focal_px or 0:.0f} px, "
        f"phone focal {phone_focal(image):.0f} px"
    )
    for source, focal in (("DA3", pred.focal_px), ("phone", phone_focal(image))):
        if focal is None:
            continue
        masks = [r.mask for r in regions]
        m = measure_food_volumes(image, plate, masks, estimator=estimator, focal_px=focal)
        if m is None:
            print(f"  {source}: plate cannot set the scale (clipped or not round)")
            continue
        print(
            f"  {source}: plate {m.cm_per_px * 10:.3f} mm/px, "
            f"camera {m.plate_distance_cm:.0f} cm above the plate, depth scale x{m.depth_scale:.3g}"
        )
        for region, ml in zip(regions, m.volumes_ml, strict=True):
            grams = estimate_grams(max(ml, 0.0), region.category)
            detail = matcher.explain(image, region) if source == "DA3" else None
            dish = detail.match.dish_name if detail else ""
            why = detail.reason.value if detail else ""
            print(
                f"    {region.category.value:24s} {ml:7.1f} ml  {grams.grams:6.0f} g "
                f"[{grams.grams_low:.0f}-{grams.grams_high:.0f}]  {dish or ''} {why}"
            )
            rows.append((name, source, region.category.value, ml, grams.grams, dish))

## Look at it: masks, depth, and the height map
The bare plate should be flat (height near 0 mm); food should stand up from it. If the plate
itself shows a bowl shape or a tilt, the plane fit or the depth model is the problem.

In [ ]:
from depth.plane import fit_plane, height_map

BLUE, RED = np.array([255, 120, 60]), np.array([40, 40, 255])
for name in NAMES[:4]:
    path = next(Path("/content/photos").rglob(name), None)
    if path is None:
        continue
    image = cv2.imread(str(path))
    found, plate = segmenter.segment(image)
    if plate is None:
        continue
    depth = estimator.predict(image)
    food = np.zeros(plate.shape, bool)
    for r in found:
        food |= r.mask
    bare = plate & ~food
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    over = image.copy()
    over[bare] = (0.6 * over[bare] + 0.4 * BLUE).astype(np.uint8)
    over[food] = (0.4 * over[food] + 0.6 * RED).astype(np.uint8)
    ax[0].imshow(over[:, :, ::-1])
    ax[0].set_title(name)
    ax[1].imshow(depth, cmap="magma")
    ax[1].set_title("DA3 depth (relative)")
    try:
        h = height_map(depth, fit_plane(depth, bare))
        lim = np.percentile(np.abs(h[plate]), 99)
        ax[2].imshow(np.where(plate, h, np.nan), cmap="coolwarm", vmin=-lim, vmax=lim)
        ax[2].set_title("height above plate plane (model units)")
    except Exception as exc:  # show the failure instead of hiding it
        ax[2].set_title(f"plane fit failed: {exc}")
    for a in ax:
        a.axis("off")
    plt.show()

Send back: the printed numbers from the run cell and the figures. What to look for:
* empty plate: every class ~0 ml (or no food classes at all);
* camera distance: hand-held phone over a plate is roughly 25-60 cm;
* the DA3 vs phone focal rows: how far apart the gram estimates are (that gap is our
  calibration uncertainty, and fixing the rig's focal length removes it);
* chicken photos: which Village chicken dish SigLIP picked, or `low_margin` / `low_probability`.